# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdul-Samad-17/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page (content_hash_id), for one client (client_hash_id), on one day (report_date) — the grain of fact_content_daily_performance.

Time window: developing on one mid-panel month, month=2026-03, per the guide's recommendation — avoids Hugging Face rate limits and avoids the _sample table's future-peeking risk.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Setup: DuckDB + Hugging Face authentication (all in one) ---
import os
import duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

# Register the Hugging Face token as a DuckDB secret
con.sql(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{os.environ["HF_TOKEN"]}'
    );
""")

BASE = "hf://datasets/FlyRank/internship-warehouse"

dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as n
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
""").df()

print("Duplicate grain rows:", len(dupes))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (observable, knowable at decision time): impressions, clicks, avg_position, ctr (derived from clicks/impressions), sessions.

Label/proxy: a decline flag — whether a page's traffic drops in a future window relative to a prior window. Not yet defined precisely; will build this in Part 3 of a later notebook with a strict prior→future split.

Context (not features, not labels — background only): client_hash_id, content_hash_id, report_date, ga4_data_available (used to check validity of a row, not fed to the model).

Excluded: any FlyRank product decision flag (health_score, priority_score, action_type, refresh tier). Why: these aren't in the release on purpose — using them as a feature would mean training a model to copy the product's own existing decision instead of discovering real signal, which is circular and teaches nothing.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Row count and date span
summary = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as min_date,
        MAX(report_date) as max_date,
        COUNT(DISTINCT client_hash_id) as n_clients,
        COUNT(DISTINCT content_hash_id) as n_content
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(summary)

   total_rows   min_date   max_date  n_clients  n_content
0     9841378 2026-03-01 2026-03-31         55     331437


In [7]:
# Availability — filter with IS TRUE, show survival rate
avail = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(avail)
print("Survival rate:", round(avail['ga4_available_rows'][0] / avail['total_rows'][0], 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0
Survival rate: 0.042


In [10]:
# Missing values check on planned features (corrected column names)
missing = con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) as missing_impressions,
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) as missing_clicks,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) as missing_position,
        SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) as missing_sessions,
        COUNT(*) as total_rows
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(missing)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   missing_impressions  missing_clicks  missing_position  missing_sessions  \
0                  0.0             0.0         6230317.0         3018741.0   

   total_rows  
0     9841378  


## 3. Verify it with queries (grain, counts, missing values, windows)

Grain confirmed: zero duplicate rows for (date, client, content) — one row genuinely is one page × one client × one day.

Row count and date span: 9,841,378 rows for March 2026, covering 2026-03-01 to 2026-03-31, across 55 clients and 331,437 unique content items.

Availability: only 413,966 of 9,841,378 rows (4.2%) have ga4_data_available IS TRUE. This confirms the guide's warning — most rows in this snapshot are search-only (GSC) with no GA4 session tracking yet, and must not be treated as "zero traffic."

Missing values: gsc_impressions and gsc_clicks have zero nulls (fully populated) — but gsc_avg_position is missing in 6,230,317 rows (63.3%) and session-related fields are missing in 3,018,741 rows (30.7%). This is consistent with the availability finding: position is presumably null when a page had no ranking/impressions that day, and session data is null for the ~96% of rows without GA4 tracking.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced history: only a minority of clients likely have 12+ months of data (to be confirmed with a lighter cross-month check) — this snapshot alone has 55 clients for March 2026, but the full guide states only 9 of 70 total clients have 12+ months, so seasonality analysis isn't reliable for most clients.

GSC-only early rows / low GA4 coverage: in this March slice, only 4.2% of rows have GA4 data available — meaning the vast majority of rows are search-visibility-only. Any feature relying on sessions/engagement must explicitly filter or flag these rows rather than assuming zero engagement.

Position often missing: gsc_avg_position is null in 63.3% of rows — likely pages with zero impressions that day, meaning ranking-based features need a real handling strategy (exclude vs. impute) rather than blindly filling with 0.

Window overlaps: not yet tested here, but any future prior→future label design must ensure the feature window and target window never share days — this is the next check before building a real predictive label.

Not causal: this data can show association only, never proof that a refresh action causes recovery.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quantify unbalanced history across clients
clients = con.sql(f"""
    SELECT client_hash_id,
           MIN(report_date) as first_date,
           MAX(report_date) as last_date,
           DATE_DIFF('month', MIN(report_date), MAX(report_date)) as months_covered
    FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
    GROUP BY client_hash_id
""").df()

print("Clients with 12+ months history:", (clients["months_covered"] >= 12).sum(), "out of", len(clients))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients with 12+ months history: 9 out of 70


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.